# LAB 02 — ELT Pipeline: Ingestion → Bronze → Silver

**Databricks Fundamentals | ~75 min | Difficulty ★★★★☆**

---

## Scenario

You are a data engineer at **RetailHub**, a mid-size e-commerce company.
Three raw data files have landed on a shared Unity Catalog Volume:

| File | Format | Contents |
|------|--------|----------|
| `customers/customers.csv` | CSV | ~1 000 customer records |
| `orders/orders_batch.json` | JSON | ~5 000 order records |
| `products/products.csv` | CSV | product catalog |

Your task: build a **Bronze → Silver ELT pipeline** that:
1. Reads each file with an **explicit schema** (no inferSchema in production)
2. Adds audit metadata (`_load_ts`, `_source_file`)
3. Cleans, casts, enriches and joins the data
4. Writes typed Silver Delta tables
5. Creates a reporting view for the analytics team

## Task map

| # | Title | Type | Difficulty |
|---|-------|------|-----------|
| 01 | Explore the Volume | GIVEN | ★ |
| 02 | Read CSV — spot schema problems | GIVEN | ★ |
| 03 | Define StructType for customers | **TODO** | ★★ |
| 04 | Read JSON orders with explicit schema | **TODO** | ★★ |
| 05 | Add audit metadata columns | **TODO** | ★★ |
| 06 | Write Bronze Delta tables | **TODO** | ★★ |
| 07 | Cast & clean Silver customers | **TODO** | ★★★ |
| 08 | Enrich Silver orders | **TODO** | ★★★ |
| 09 | Join orders ↔ customers | **TODO** | ★★★ |
| 10 | Aggregate: revenue per customer | **TODO** | ★★★ |
| 11 | Write Silver Delta table (partitioned) | **TODO** | ★★★★ |
| 12 | Temp view + ad-hoc SQL analysis | **TODO** | ★★★ |

In [ ]:
# Setup — run this first
%run ../setup/00_setup

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
    # Task 01 — Explore the Volume                                      [GIVEN]
    # ─────────────────────────────────────────────────────────────────────────
    # dbutils.fs.ls() lists files on a DBFS/Volume path.

    print("=== datasets root ===")
    for f in dbutils.fs.ls(DATASET_PATH):
        print(f"  {f.name:40s}  {f.size:>10,} bytes")

    print("
=== orders/ ===")
    for f in dbutils.fs.ls(f"{DATASET_PATH}/orders"):
        print(f"  {f.name:40s}  {f.size:>10,} bytes")

    # ❓ How many files are in orders/?
    # ❓ Is the file size of orders_batch.json reasonable for ~5 000 rows?

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# Task 02 — Read customers.csv: auto-schema, spot the problems      [GIVEN]
# ─────────────────────────────────────────────────────────────────────────

df_infer = (
    spark.read
         .option("header", "true")
         .option("inferSchema", "true")
         .csv(f"{DATASET_PATH}/customers/customers.csv")
)

print("=== Schema inferred by Spark ===")
df_infer.printSchema()
df_infer.show(5)

# ❓ What type did Spark assign to registration_date? Is it correct?
# ❓ What type is loyalty_points? What would happen if a null slips in?
# ❓ Why is inferSchema risky in a production pipeline?

### Task 03 — Define StructType for customers  `[TODO]`

The inferred schema above uses **StringType** for dates and may guess
numeric types incorrectly. Define a manual `StructType` and re-read the file.

**API hint — constructing a schema:**
```python
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, DateType

schema = StructType([
    StructField("column_name", DataType(), nullable=True),
    ...
])

df = (spark.read
           .schema(schema)
           .option("header", "true")
           .option("dateFormat", "yyyy-MM-dd")
           .csv(path))
```

**Customers CSV columns:**
`customer_id` (String, NOT NULL), `first_name` (String), `last_name` (String),
`email` (String), `country` (String), `registration_date` (**DateType**),
`loyalty_points` (**IntegerType**), `annual_spend` (**DoubleType**)

In [ ]:
# Task 03 — TODO: implement manual schema + re-read

from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, DateType

# TODO: define customer_schema
customer_schema = StructType([
    # StructField("customer_id",      StringType(),  False),
    # StructField("first_name",       StringType(),  True),
    # StructField("last_name",        StringType(),  True),
    # StructField("email",            StringType(),  True),
    # StructField("country",          StringType(),  True),
    # StructField("registration_date",DateType(),    True),
    # StructField("loyalty_points",   IntegerType(), True),
    # StructField("annual_spend",     DoubleType(),  True),
])

# TODO: re-read customers.csv using customer_schema
df_customers_raw = (
    spark.read
    # .schema(customer_schema)
    # .option("header", "true")
    # .option("dateFormat", "yyyy-MM-dd")
    # .csv(f"{DATASET_PATH}/customers/customers.csv")
)

print("=== Manual schema ===")
df_customers_raw.printSchema()
print(f"Row count: {df_customers_raw.count()}")
df_customers_raw.show(5)

# Self-check
assert dict(df_customers_raw.dtypes)["registration_date"] == "date",         "registration_date must be DateType"
assert dict(df_customers_raw.dtypes)["loyalty_points"] == "int",         "loyalty_points must be IntegerType"
print("✅ Schema assertions passed")

### Task 04 — Read JSON orders with explicit schema  `[TODO]`

`inferSchema` on JSON reads the **entire file** to detect types — expensive on large files.
A manual schema lets you control `TimestampType` for datetime fields.

**API hint — JSON with explicit schema:**
```python
from pyspark.sql.types import TimestampType

df = (
    spark.read
         .schema(orders_schema)
         .option("timestampFormat", "yyyy-MM-dd HH:mm:ss")
         .json(path)
)
```

**Orders JSON fields:**
`order_id` (String), `customer_id` (String), `product_id` (String),
`store_id` (String), `order_datetime` (**TimestampType**),
`quantity` (IntegerType), `unit_price` (DoubleType),
`discount_percent` (DoubleType), `total_amount` (DoubleType),
`payment_method` (String), `status` (String)

In [ ]:
# Task 04 — TODO: define orders schema + read JSON

from pyspark.sql.types import (
    StructType, StructField,
    StringType, IntegerType, DoubleType, TimestampType
)

# TODO: define orders_schema
orders_schema = StructType([
    # StructField("order_id",         StringType(),    False),
    # StructField("customer_id",      StringType(),    True),
    # StructField("product_id",       StringType(),    True),
    # StructField("store_id",         StringType(),    True),
    # StructField("order_datetime",   TimestampType(), True),
    # StructField("quantity",         IntegerType(),   True),
    # StructField("unit_price",       DoubleType(),    True),
    # StructField("discount_percent", DoubleType(),    True),
    # StructField("total_amount",     DoubleType(),    True),
    # StructField("payment_method",   StringType(),    True),
    # StructField("status",           StringType(),    True),
])

# TODO: read orders_batch.json using the explicit schema
df_orders_raw = (
    spark.read
    # .schema(orders_schema)
    # .option("timestampFormat", "yyyy-MM-dd HH:mm:ss")
    # .json(f"{DATASET_PATH}/orders/orders_batch.json")
)

print("=== Orders schema ===")
df_orders_raw.printSchema()
print(f"Row count: {df_orders_raw.count()}")
df_orders_raw.show(5, truncate=False)

assert dict(df_orders_raw.dtypes)["order_datetime"] == "timestamp",         "order_datetime must be TimestampType"
assert dict(df_orders_raw.dtypes)["total_amount"] == "double",         "total_amount must be DoubleType"
print("✅ Schema assertions passed")

### Task 05 — Add audit metadata columns  `[TODO]`

Every Bronze table must record **when** and **from where** each row was loaded.
This is critical for debugging incremental loads.

**API hint:**
```python
from pyspark.sql import functions as F

df = (
    df
    .withColumn("_load_ts",     F.current_timestamp())   # TimestampType
    .withColumn("_source_file", F.lit("path/to/file"))   # StringType literal
)
```

In [ ]:
# Task 05 — TODO: add _load_ts and _source_file to BOTH DataFrames

from pyspark.sql import functions as F

CUSTOMERS_PATH = f"{DATASET_PATH}/customers/customers.csv"
ORDERS_PATH    = f"{DATASET_PATH}/orders/orders_batch.json"

# TODO: enrich df_customers_raw → df_customers_bronze
df_customers_bronze = (
    df_customers_raw
    # .withColumn("_load_ts",     F.current_timestamp())
    # .withColumn("_source_file", F.lit(CUSTOMERS_PATH))
)

# TODO: enrich df_orders_raw → df_orders_bronze
df_orders_bronze = (
    df_orders_raw
    # .withColumn("_load_ts",     F.current_timestamp())
    # .withColumn("_source_file", F.lit(ORDERS_PATH))
)

print("Customers columns:", df_customers_bronze.columns)
print("Orders columns:   ", df_orders_bronze.columns)

assert "_load_ts"     in df_customers_bronze.columns, "_load_ts missing from customers"
assert "_source_file" in df_orders_bronze.columns,   "_source_file missing from orders"
print("✅ Audit columns present")

### Task 06 — Write Bronze Delta tables  `[TODO]`

Write both enriched DataFrames to managed Delta tables in the `bronze` schema.
Use **overwrite** mode — Bronze is always a full reload from the source file.

**API hint — writing a managed Delta table (Unity Catalog):**
```python
(
    df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{CATALOG}.{SCHEMA}.table_name")
)
```

Target tables:
- `{CATALOG}.bronze.lab_customers`
- `{CATALOG}.bronze.lab_orders`

In [ ]:
# Task 06 — TODO: write Bronze Delta tables

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.bronze")

# TODO: write df_customers_bronze → {CATALOG}.bronze.lab_customers


# TODO: write df_orders_bronze → {CATALOG}.bronze.lab_orders


# Verify
rows_c = spark.table(f"{CATALOG}.bronze.lab_customers").count()
rows_o = spark.table(f"{CATALOG}.bronze.lab_orders").count()
print(f"bronze.lab_customers rows: {rows_c:,}")
print(f"bronze.lab_orders rows:    {rows_o:,}")

### Task 07 — Cast & clean Silver customers  `[TODO]`

Silver is **typed, clean and enriched**. Apply the following rules:

1. Keep only: `customer_id`, `first_name`, `last_name`, `email`, `country`,
   `registration_date`, `loyalty_points`, `annual_spend`
2. Drop rows where `customer_id` IS NULL or empty string
3. Normalise `country` to UPPER CASE
4. Add `full_name = first_name + " " + last_name`
5. Add `loyalty_tier`:
   - `"gold"` if `loyalty_points >= 1000`
   - `"silver"` if `loyalty_points >= 500`
   - `"bronze"` otherwise

**API hints:**
```python
# conditional column
F.when(condition, value).when(condition2, value2).otherwise(default)

# string concat
F.concat_ws(" ", F.col("first_name"), F.col("last_name"))

# null / empty guard
F.col("x").isNotNull() & (F.col("x") != "")

# upper case
F.upper(F.col("country"))
```

In [ ]:
# Task 07 — TODO: build Silver customers DataFrame

from pyspark.sql import functions as F

df_silver_customers = (
    spark.table(f"{CATALOG}.bronze.lab_customers")
    # TODO 1: .select(...)                                  ← keep listed columns only
    # TODO 2: .filter(...)                                  ← drop null/empty customer_id
    # TODO 3: .withColumn("country",       F.upper(...))
    # TODO 4: .withColumn("full_name",     F.concat_ws(...))
    # TODO 5: .withColumn("loyalty_tier",  F.when(...).when(...).otherwise("bronze"))
)

df_silver_customers.printSchema()
df_silver_customers.show(10, truncate=False)
print(f"Silver customers row count: {df_silver_customers.count()}")

# Spot-check loyalty_tier distribution
df_silver_customers.groupBy("loyalty_tier").count().orderBy("count", ascending=False).show()

### Task 08 — Enrich Silver orders  `[TODO]`

Business rules for `df_silver_orders`:

1. Keep only orders with `status` IN (`'shipped'`, `'delivered'`, `'returned'`)
2. Extract `order_date` (DateType) and `order_hour` (IntegerType) from `order_datetime`
3. Calculate `revenue_net = total_amount * (1 - discount_percent / 100.0)`
4. Add boolean `is_discounted = discount_percent > 0`

**API hints:**
```python
F.col("status").isin(["shipped", "delivered", "returned"])
F.to_date(F.col("order_datetime"))
F.hour(F.col("order_datetime"))
F.col("discount_percent") > 0
```

In [ ]:
# Task 08 — TODO: build Silver orders DataFrame

from pyspark.sql import functions as F

df_silver_orders = (
    spark.table(f"{CATALOG}.bronze.lab_orders")
    # TODO 1: .filter(F.col("status").isin([...]))
    # TODO 2: .withColumn("order_date",    F.to_date(F.col("order_datetime")))
    # TODO 3: .withColumn("order_hour",    F.hour(F.col("order_datetime")))
    # TODO 4: .withColumn("revenue_net",   <expression>)
    # TODO 5: .withColumn("is_discounted", <boolean expression>)
)

df_silver_orders.printSchema()
df_silver_orders.show(5, truncate=False)
print(f"Silver orders row count: {df_silver_orders.count()}")

# Verify 'pending' and 'cancelled' are excluded
df_silver_orders.groupBy("status").count().show()

### Task 09 — Join orders ↔ customers  `[TODO]`

Produce a flat enriched DataFrame combining order and customer dimensions.

- Join `df_silver_orders` (left) with `df_silver_customers` (right) on `customer_id`
- Use a **left join** — preserve all orders even without a matching customer

**API hint:**
```python
df_joined = df_left.join(
    df_right,
    on  = "join_key",   # column name, list, or boolean
    how = "left"        # inner | left | right | full | semi | anti
)
```

Keep only:
`order_id`, `order_date`, `order_hour`, `status`, `revenue_net`,
`is_discounted`, `payment_method`, `customer_id`,
`full_name`, `country`, `loyalty_tier`

In [ ]:
# Task 09 — TODO: join orders and customers

from pyspark.sql import functions as F

df_joined = (
    df_silver_orders
    # .join(df_silver_customers, on="customer_id", how="left")
    # .select("order_id","order_date","order_hour","status","revenue_net",
    #         "is_discounted","payment_method","customer_id",
    #         "full_name","country","loyalty_tier")
)

print(f"Joined row count: {df_joined.count()}")
df_joined.show(10, truncate=False)

no_customer = df_joined.filter(F.col("full_name").isNull()).count()
print(f"Orders without a matching customer record: {no_customer}")

### Task 10 — Aggregate: revenue per customer  `[TODO]`

Produce a Gold-style customer summary.

| Output column | How to compute |
|---|---|
| `customer_id` | group key |
| `full_name` | group key |
| `country` | group key |
| `loyalty_tier` | group key |
| `total_orders` | `F.count("order_id")` |
| `total_revenue` | `F.sum("revenue_net")` rounded 2 dp |
| `avg_order_value` | `F.mean("revenue_net")` rounded 2 dp |
| `first_order_date` | `F.min("order_date")` |
| `last_order_date` | `F.max("order_date")` |

Sort by `total_revenue` descending.

**API hint:**
```python
df.groupBy("a","b").agg(
    F.count("x").alias("cnt"),
    F.round(F.sum("y"), 2).alias("total")
).orderBy(F.col("total").desc())
```

In [ ]:
# Task 10 — TODO: aggregate revenue per customer

from pyspark.sql import functions as F

df_customer_summary = (
    df_joined
    # .groupBy("customer_id","full_name","country","loyalty_tier")
    # .agg(
    #     F.count("order_id").alias("total_orders"),
    #     F.round(F.sum("revenue_net"),  2).alias("total_revenue"),
    #     F.round(F.mean("revenue_net"), 2).alias("avg_order_value"),
    #     F.min("order_date").alias("first_order_date"),
    #     F.max("order_date").alias("last_order_date"),
    # )
    # .orderBy(F.col("total_revenue").desc())
)

df_customer_summary.show(20, truncate=False)
print(f"Distinct customers in summary: {df_customer_summary.count()}")

### Task 11 — Write Silver Delta table (partitioned)  `[TODO]`

Write `df_silver_orders` to `{CATALOG}.silver.lab_orders` **partitioned by `order_date`**
so daily queries skip irrelevant partitions.

**API hint — partitioned write:**
```python
(
    df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .partitionBy("partition_column")
    .saveAsTable(f"{CATALOG}.silver.table_name")
)
```

Also write `df_silver_customers` to `{CATALOG}.silver.lab_customers`
(no partitioning).

In [ ]:
# Task 11 — TODO: write Silver Delta tables

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.silver")

# TODO: write df_silver_orders, partitioned by order_date


# TODO: write df_silver_customers (no partition)


# Verify
for tbl in [f"{CATALOG}.silver.lab_orders", f"{CATALOG}.silver.lab_customers"]:
    cnt    = spark.table(tbl).count()
    detail = spark.sql(f"DESCRIBE DETAIL {tbl}")                       .select("numFiles","partitionColumns").collect()[0]
    print(f"{tbl}")
    print(f"  rows={cnt:,}  files={detail['numFiles']}  partitions={detail['partitionColumns']}")

### Task 12 — Temp view + ad-hoc SQL analysis  `[TODO]`

Register `df_customer_summary` as a temporary view and answer four
business questions using pure SQL.

**API hint:**
```python
df.createOrReplaceTempView("view_name")
spark.sql("SELECT ... FROM view_name ...").show()
```

**Questions:**
1. Which country has the highest total revenue?
2. What percentage of total revenue comes from `gold` loyalty-tier customers?
3. What is the average number of orders per loyalty tier?
4. Which single customer placed the most orders?

In [ ]:
# Task 12 — TODO: register temp view + run 4 SQL queries

# TODO: register df_customer_summary as 'customer_summary'
# df_customer_summary.createOrReplaceTempView('customer_summary')

# Q1 — country with highest total revenue
print('Q1 — Revenue by country:')
# spark.sql(
#     'SELECT country, ROUND(SUM(total_revenue),2) AS country_revenue'
#     ' FROM customer_summary GROUP BY country ORDER BY country_revenue DESC'
# ).show()

# Q2 — % revenue from gold tier
print('\nQ2 — Gold tier revenue share:')
# spark.sql(
#     'SELECT loyalty_tier,'
#     ' ROUND(100.0 * SUM(total_revenue) / SUM(SUM(total_revenue)) OVER (), 2) AS pct'
#     ' FROM customer_summary GROUP BY loyalty_tier ORDER BY pct DESC'
# ).show()

# Q3 — avg orders per loyalty tier
print('\nQ3 — Avg orders per loyalty tier:')
# spark.sql(
#     'SELECT loyalty_tier, ROUND(AVG(total_orders),2) AS avg_orders'
#     ' FROM customer_summary GROUP BY loyalty_tier ORDER BY avg_orders DESC'
# ).show()

# Q4 — top customer by order count
print('\nQ4 — Top customer by order count:')
# spark.sql(
#     'SELECT customer_id, full_name, total_orders'
#     ' FROM customer_summary ORDER BY total_orders DESC LIMIT 1'
# ).show()